# Mean HbA1c Sensitivity Analysis

**Purpose:** Re-compute baseline and follow-up HbA1c using the **mean** of all values
in the window (instead of the most recent single value), then re-run DML as a
sensitivity analysis.

**Why:** The primary analysis uses the last (most recent) A1c value in each window.
Aaron Baum suggested using the mean of all available values to reduce measurement noise
and improve precision.

**Run this notebook on SageMaker** (requires database access via SSM).

In [ ]:
from __future__ import annotations

import json
import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import text

warnings.filterwarnings("ignore")

# SageMaker shared lib
SAGEMAKER_LIB = Path("~/sagemaker-lib").expanduser()
if SAGEMAKER_LIB.exists() and str(SAGEMAKER_LIB) not in sys.path:
    sys.path.insert(0, str(SAGEMAKER_LIB))

try:
    import waymark
except ImportError as exc:
    raise ImportError(
        "Unable to import the internal `waymark` module. "
        "Mount ~/sagemaker-lib or install the waymark-platform wheel."
    ) from exc

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("mean_a1c")

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
BASELINE_MONTHS = 6
FOLLOWUP_MONTHS = 6
A1C_PLAUSIBLE_RANGE = (3.0, 20.0)

OUTPUT_DIR = Path("~/SageMaker/mean_a1c_sensitivity").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Path to existing analytic cohort (upload to SageMaker first)
COHORT_PARQUET = Path("~/SageMaker/diabetes/analytic_cohort.parquet").expanduser()
if not COHORT_PARQUET.exists():
    raise FileNotFoundError(
        "Upload analytic_cohort.parquet to ~/SageMaker/diabetes/ before running. "
        "Source: data/analytic_cohort.parquet in the repository."
    )

print(f"Output dir: {OUTPUT_DIR}")
print(f"Cohort parquet: {COHORT_PARQUET}")

In [ ]:
# ── Connect to database ──────────────────────────────────────────
engine = waymark.get_waymark_core_db_engine()
logger.info("Database connection OK")

In [ ]:
# ── Load existing analytic cohort ────────────────────────────────
cohort = pd.read_parquet(COHORT_PARQUET)
print(f"Analytic cohort: {len(cohort)} rows")
print(f"  Treated: {cohort['treated'].sum()}")
print(f"  Control: {(cohort['treated'] == 0).sum()}")
print(f"  Columns: {list(cohort.columns)}")

# Check required columns (way_id = Tuva person_id, no member_id in this cohort)
for col in ["way_id", "index_date", "treated", "baseline_a1c", "followup_a1c", "a1c_change"]:
    assert col in cohort.columns, f"Missing column: {col}"
print("All required columns present.")

In [ ]:
# ── Step 1: Extract ALL A1c labs from database ───────────────────
query = """
    SELECT
        person_id,
        collection_date,
        result
    FROM dbt_tuva_core.lab_result
    WHERE source_description ILIKE '%%a1c%%'
      AND translate(result, '<>', '') ~ '^\\d+(\\.\\d+)?$'
"""
raw = pd.read_sql(query, engine)
print(f"Raw A1c rows: {len(raw)}")

raw["a1c_value"] = (
    raw["result"]
    .str.replace("<", "", regex=False)
    .str.replace(">", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)
raw["collection_date"] = pd.to_datetime(raw["collection_date"], errors="coerce")
raw = raw[raw["a1c_value"].between(*A1C_PLAUSIBLE_RANGE)].copy()

# Deduplicate: if same patient has multiple values on same day, take min
a1c = (
    raw.groupby(["person_id", "collection_date"])
    .agg(a1c_value=("a1c_value", "min"))
    .reset_index()
)
print(f"Cleaned A1c: {len(a1c)} person-days, {a1c['person_id'].nunique()} patients")
print(f"Mean A1c: {a1c['a1c_value'].mean():.1f}, Median: {a1c['a1c_value'].median():.1f}")

In [ ]:
# ── Step 2: Filter A1c labs to cohort members ───────────────────
# way_id in the cohort IS the Tuva person_id, so match directly
cohort_ids = set(cohort["way_id"].dropna().unique())
a1c = a1c[a1c["person_id"].isin(cohort_ids)].copy()
print(f"A1c values matched to cohort: {len(a1c)} rows, {a1c['person_id'].nunique()} patients")

In [ ]:
# ── Step 3: Compute MEAN A1c in baseline and follow-up windows ──
# Merge index_date from cohort (join on person_id = way_id)
a1c = a1c.merge(
    cohort[["way_id", "index_date"]].rename(columns={"way_id": "person_id"}),
    on="person_id", how="inner",
)
a1c["index_date"] = pd.to_datetime(a1c["index_date"])

a1c["baseline_start"] = a1c["index_date"] - pd.DateOffset(months=BASELINE_MONTHS)
a1c["followup_end"] = a1c["index_date"] + pd.DateOffset(months=FOLLOWUP_MONTHS)

# Baseline window: [index_date - 6 months, index_date)
baseline_mask = (
    (a1c["collection_date"] >= a1c["baseline_start"]) &
    (a1c["collection_date"] < a1c["index_date"])
)
baseline_vals = a1c[baseline_mask].copy()

# Follow-up window: [index_date, index_date + 6 months]
followup_mask = (
    (a1c["collection_date"] >= a1c["index_date"]) &
    (a1c["collection_date"] <= a1c["followup_end"])
)
followup_vals = a1c[followup_mask].copy()

# --- LAST value (for verification against existing data) ---
baseline_last = (
    baseline_vals.sort_values("collection_date")
    .groupby("person_id")
    .last()[["a1c_value", "collection_date"]]
    .rename(columns={"a1c_value": "baseline_a1c_last", "collection_date": "baseline_a1c_date_last"})
)
followup_last = (
    followup_vals.sort_values("collection_date")
    .groupby("person_id")
    .last()[["a1c_value", "collection_date"]]
    .rename(columns={"a1c_value": "followup_a1c_last", "collection_date": "followup_a1c_date_last"})
)

# --- MEAN value (the new sensitivity measure) ---
baseline_mean = (
    baseline_vals.groupby("person_id")
    .agg(
        baseline_a1c_mean=("a1c_value", "mean"),
        baseline_a1c_count=("a1c_value", "count"),
    )
)
followup_mean = (
    followup_vals.groupby("person_id")
    .agg(
        followup_a1c_mean=("a1c_value", "mean"),
        followup_a1c_count=("a1c_value", "count"),
    )
)

print(f"Baseline: {len(baseline_last)} patients with last value, {len(baseline_mean)} with mean")
print(f"Follow-up: {len(followup_last)} patients with last value, {len(followup_mean)} with mean")

# Count distribution
print(f"\nBaseline A1c values per patient:")
print(baseline_mean["baseline_a1c_count"].describe())
print(f"\nFollow-up A1c values per patient:")
print(followup_mean["followup_a1c_count"].describe())

In [ ]:
# ── Step 4: Merge into cohort ───────────────────────────────────
# All groupby results are indexed by person_id; rename to way_id for merge
baseline_last.index.name = "way_id"
followup_last.index.name = "way_id"
baseline_mean.index.name = "way_id"
followup_mean.index.name = "way_id"

cohort_aug = cohort.copy()
cohort_aug = cohort_aug.merge(baseline_last, on="way_id", how="left")
cohort_aug = cohort_aug.merge(followup_last, on="way_id", how="left")
cohort_aug = cohort_aug.merge(baseline_mean, on="way_id", how="left")
cohort_aug = cohort_aug.merge(followup_mean, on="way_id", how="left")

# Compute mean-based A1c change
cohort_aug["a1c_change_mean"] = cohort_aug["followup_a1c_mean"] - cohort_aug["baseline_a1c_mean"]

# Verify: last-value matches existing data
has_both = cohort_aug["baseline_a1c_last"].notna() & cohort_aug["baseline_a1c"].notna()
corr = cohort_aug.loc[has_both, "baseline_a1c_last"].corr(cohort_aug.loc[has_both, "baseline_a1c"])
max_diff = (cohort_aug.loc[has_both, "baseline_a1c_last"] - cohort_aug.loc[has_both, "baseline_a1c"]).abs().max()
print(f"Verification: baseline_a1c_last vs existing baseline_a1c")
print(f"  Correlation: {corr:.4f}")
print(f"  Max absolute difference: {max_diff:.4f}")

# Summary
has_mean = cohort_aug["baseline_a1c_mean"].notna() & cohort_aug["followup_a1c_mean"].notna()
has_last = cohort_aug["baseline_a1c"].notna() & cohort_aug["followup_a1c"].notna()
print(f"\nPatients with both baseline+followup (last): {has_last.sum()}")
print(f"Patients with both baseline+followup (mean): {has_mean.sum()}")

# Compare mean vs last
both = cohort_aug[has_mean & has_last].copy()
print(f"\nMean vs Last baseline A1c: r={both['baseline_a1c_mean'].corr(both['baseline_a1c']):.3f}")
print(f"Mean vs Last followup A1c: r={both['followup_a1c_mean'].corr(both['followup_a1c']):.3f}")
print(f"Mean A1c change (last-based): {both['a1c_change'].mean():.3f}")
print(f"Mean A1c change (mean-based): {both['a1c_change_mean'].mean():.3f}")

In [ ]:
# ── Step 5: Save augmented cohort ────────────────────────────────
out_parquet = OUTPUT_DIR / "analytic_cohort_mean_a1c.parquet"
cohort_aug.to_parquet(out_parquet, index=False)
print(f"Saved augmented cohort to {out_parquet}")
print(f"New columns: baseline_a1c_mean, followup_a1c_mean, a1c_change_mean, baseline_a1c_count, followup_a1c_count")

---
## DML Sensitivity Analysis with Mean A1c

Re-run the primary DML analysis using mean A1c values instead of last values.

In [ ]:
from econml.dml import LinearDML, CausalForestDML
from lightgbm import LGBMRegressor
from sklearn.linear_model import LogisticRegression
from scipy import stats

# ── Covariates (same as reanalysis.py) ────────────────────────────
COVARIATES = [
    "age", "baseline_a1c", "risk_percentile", "comorbidity_count",
    "pre_ed", "pre_ip", "pre_pcp", "has_bh", "has_htn",
    "has_chf", "has_pulm", "polypharmacy", "high_ed_ip",
]

# For the mean-A1c analysis, replace baseline_a1c with baseline_a1c_mean in covariates
COVARIATES_MEAN = [
    "age", "baseline_a1c_mean", "risk_percentile", "comorbidity_count",
    "pre_ed", "pre_ip", "pre_pcp", "has_bh", "has_htn",
    "has_chf", "has_pulm", "polypharmacy", "high_ed_ip",
]

PS_LO, PS_HI = 0.05, 0.95


def run_dml(Y, T, W, label=""):
    """Run LinearDML, return dict of results."""
    dml = LinearDML(
        model_y=LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, verbose=-1),
        model_t=LogisticRegression(max_iter=1000),
        discrete_treatment=True, cv=5, random_state=42,
    )
    dml.fit(Y, T, X=None, W=W)
    ate = float(dml.ate())
    ci = dml.ate_interval(alpha=0.05)
    ci_lo, ci_hi = float(ci[0]), float(ci[1])
    pooled_sd = float(np.sqrt((Y[T == 1].var() + Y[T == 0].var()) / 2))
    d = abs(ate) / pooled_sd if pooled_sd > 0 else 0
    se = (ci_hi - ci_lo) / (2 * 1.96)
    z = ate / se if se > 0 else 0
    p = float(2 * (1 - stats.norm.cdf(abs(z))))
    print(f"  [{label}] ATE={ate:.3f} (95% CI: {ci_lo:.3f} to {ci_hi:.3f}), d={d:.3f}, P={p:.4f}")
    return {
        "label": label, "ate": round(ate, 4), "ci_lo": round(ci_lo, 4),
        "ci_hi": round(ci_hi, 4), "cohens_d": round(d, 3), "p_value": round(p, 4),
        "n": int(len(Y)), "n_treated": int(T.sum()), "n_control": int((1 - T).sum()),
        "pooled_sd": round(pooled_sd, 4),
    }


print("DML helper loaded")

In [ ]:
# ── Analysis 1: DML with mean A1c outcome on full cohort ────────
print("=== DML with mean A1c outcome (full cohort) ===")

df = cohort_aug.dropna(subset=COVARIATES_MEAN + ["a1c_change_mean"]).copy()
print(f"N after dropping NAs: {len(df)} (treated={df['treated'].sum()}, control={(df['treated'] == 0).sum()})")

# PS trimming
X_ps = df[COVARIATES_MEAN].values.astype(float)
T_ps = df["treated"].values.astype(float)
ps = LogisticRegression(max_iter=1000).fit(X_ps, T_ps).predict_proba(X_ps)[:, 1]
mask = (ps >= PS_LO) & (ps <= PS_HI)
df_trim = df[mask].copy()
print(f"After PS trimming: {len(df_trim)}")

W = df_trim[COVARIATES_MEAN].values.astype(float)
Y = df_trim["a1c_change_mean"].values.astype(float)
T = df_trim["treated"].values.astype(float)

result_mean_full = run_dml(Y, T, W, label="DML, mean A1c, full cohort")

In [ ]:
# ── Analysis 2: DML with mean A1c, subgroup baseline_a1c_mean >= 8 ─
print("=== DML with mean A1c, subgroup baseline_a1c_mean >= 8.0 ===")

sub = df_trim[df_trim["baseline_a1c_mean"] >= 8.0].copy()
print(f"Subgroup N: {len(sub)} (treated={sub['treated'].sum()}, control={(sub['treated'] == 0).sum()})")

if sub["treated"].sum() >= 5 and (sub["treated"] == 0).sum() >= 5:
    # Re-estimate PS within subgroup
    X_ps_sub = sub[COVARIATES_MEAN].values.astype(float)
    T_ps_sub = sub["treated"].values.astype(float)
    ps_sub = LogisticRegression(max_iter=1000).fit(X_ps_sub, T_ps_sub).predict_proba(X_ps_sub)[:, 1]
    mask_sub = (ps_sub >= PS_LO) & (ps_sub <= PS_HI)
    sub = sub[mask_sub].copy()
    print(f"After PS trimming: {len(sub)}")

    W_sub = sub[COVARIATES_MEAN].values.astype(float)
    Y_sub = sub["a1c_change_mean"].values.astype(float)
    T_sub = sub["treated"].values.astype(float)
    result_mean_sub8 = run_dml(Y_sub, T_sub, W_sub, label="DML, mean A1c, baseline >= 8")
else:
    result_mean_sub8 = {"label": "DML, mean A1c, baseline >= 8", "error": "insufficient sample"}
    print("Insufficient sample")

In [ ]:
# ── Analysis 3: Compare last-value DML vs mean-value DML ────────
# Re-run last-value DML on the SAME trimmed cohort for direct comparison
print("=== Comparison: last-value DML on same trimmed cohort ===")

# Make sure we use the same cohort (has both last and mean values)
df_both = cohort_aug.dropna(subset=COVARIATES + COVARIATES_MEAN + ["a1c_change", "a1c_change_mean"]).copy()
X_ps_both = df_both[COVARIATES].values.astype(float)
T_ps_both = df_both["treated"].values.astype(float)
ps_both = LogisticRegression(max_iter=1000).fit(X_ps_both, T_ps_both).predict_proba(X_ps_both)[:, 1]
mask_both = (ps_both >= PS_LO) & (ps_both <= PS_HI)
df_both_trim = df_both[mask_both].copy()
print(f"Comparison cohort N: {len(df_both_trim)}")

# Last-value DML
W_last = df_both_trim[COVARIATES].values.astype(float)
Y_last = df_both_trim["a1c_change"].values.astype(float)
T_last = df_both_trim["treated"].values.astype(float)
result_last_comparison = run_dml(Y_last, T_last, W_last, label="DML, last A1c (comparison)")

# Mean-value DML (same cohort)
W_mean = df_both_trim[COVARIATES_MEAN].values.astype(float)
Y_mean = df_both_trim["a1c_change_mean"].values.astype(float)
T_mean = df_both_trim["treated"].values.astype(float)
result_mean_comparison = run_dml(Y_mean, T_mean, W_mean, label="DML, mean A1c (comparison)")

print(f"\n--- Summary ---")
print(f"Last A1c: ATE={result_last_comparison['ate']:.3f} (P={result_last_comparison['p_value']:.4f})")
print(f"Mean A1c: ATE={result_mean_comparison['ate']:.3f} (P={result_mean_comparison['p_value']:.4f})")

In [ ]:
# ── Save results ─────────────────────────────────────────────────
sensitivity_results = {
    "mean_a1c_full_cohort": result_mean_full,
    "mean_a1c_subgroup_ge8": result_mean_sub8,
    "comparison": {
        "last_value": result_last_comparison,
        "mean_value": result_mean_comparison,
        "note": "Both run on identical cohort for direct comparison",
    },
    "metadata": {
        "description": "Mean A1c sensitivity analysis (Baum suggestion)",
        "baseline_window": f"{BASELINE_MONTHS} months before index date",
        "followup_window": f"{FOLLOWUP_MONTHS} months after index date",
        "aggregation": "mean of all A1c values in window (vs. last/most recent)",
    },
}

out_json = OUTPUT_DIR / "mean_a1c_sensitivity_results.json"
with open(out_json, "w") as f:
    json.dump(sensitivity_results, f, indent=2, default=str)
print(f"Results saved to {out_json}")

# Also print for easy copy-paste into manuscript
print("\n" + "=" * 60)
print("FOR MANUSCRIPT:")
print("=" * 60)
r = result_mean_full
print(f"Mean-A1c DML (full cohort, N={r['n']}): ATE = {r['ate']:.2f} pp "
      f"(95% CI: {r['ci_lo']:.2f} to {r['ci_hi']:.2f}; Cohen's d = {r['cohens_d']:.2f})")
if "error" not in result_mean_sub8:
    r2 = result_mean_sub8
    print(f"Mean-A1c DML (baseline >= 8%, N={r2['n']}): ATE = {r2['ate']:.2f} pp "
          f"(95% CI: {r2['ci_lo']:.2f} to {r2['ci_hi']:.2f}; Cohen's d = {r2['cohens_d']:.2f})")

---
## Next Steps

1. Download `analytic_cohort_mean_a1c.parquet` and `mean_a1c_sensitivity_results.json` from SageMaker
2. Copy to `packaging/chw-diabetes-a1c/output/`
3. Add mean-A1c row to manuscript Table 2 and Results sensitivity section
4. Update appendix eTable with mean-A1c results